# 03 · Preprocessing: class filter, features and groups

Before modelling: remove the classes too small to model, build the two feature sets, and define the groups of related rows that cross-validation must keep together. Scaling and label encoding are not done here; both happen inside the cross-validation in `04_modeling`, fitted on training folds only.

Nothing is written to disk. `04_modeling` repeats these steps with `codon_utils.load_project`, and the last cell checks that the two agree.

In [1]:
import numpy as np
import pandas as pd

import codon_utils

In [2]:
# the repaired raw data, with decoded labels (see 01_wrangling)
codon_usage = codon_utils.read_raw('../data/raw/codon_usage.csv')
codon_usage.shape

(13028, 72)

The variables that I'd like to model as outcomes are 'Kingdom' and 'DNAtype'; I'd like to take another look at their value counts.

In [3]:
codon_usage['kingdom'].value_counts()

kingdom
bacteria         2920
virus            2832
plant            2523
vertebrate       2077
invertebrate     1345
mammal            572
bacteriophage     220
rodent            215
primate           180
archaea           126
plasmid            18
Name: count, dtype: int64

In [4]:
codon_usage['dna_type'].value_counts()

dna_type
genomic                   9267
mitochondrial             2899
chloroplast                816
plastid                     31
kinetoplast                  5
cyanelle                     2
unknown                      2
nucleomorph                  2
apicoplast                   2
secondary_endosymbiont       1
chromoplast                  1
Name: count, dtype: int64

Almost all rows are genomic, mitochondrial or chloroplast, and every kingdom has at least 126 rows except plasmids (18). The rare classes are dropped, as in 2022. The headline metric, macro-F1, weights every class equally, so a class with 1–31 rows would add a per-class score estimated from a handful of test rows; and 'plasmid' describes a kind of DNA molecule, not a group of organisms.

In [5]:
codon_usage = codon_usage[codon_usage['kingdom'] != 'plasmid']
codon_usage.shape

(13010, 72)

In [6]:
codon_usage = codon_usage[codon_usage['dna_type'].isin(['genomic', 'mitochondrial', 'chloroplast'])]
codon_usage.shape

(12964, 72)

This removed 46 rows: 44 rows of the rare named DNA types, and the two rows with code 9 ("NA" in the codebook), which the 2022 version lost later as missing values.

In [7]:
codon_usage[['kingdom', 'dna_type']].value_counts()

kingdom        dna_type     
bacteria       genomic          2918
virus          genomic          2832
vertebrate     mitochondrial    1613
plant          genomic          1523
invertebrate   genomic           922
plant          chloroplast       815
mammal         mitochondrial     470
vertebrate     genomic           464
invertebrate   mitochondrial     411
bacteriophage  genomic           220
rodent         mitochondrial     156
plant          mitochondrial     152
archaea        genomic           126
mammal         genomic           102
primate        mitochondrial      97
               genomic            83
rodent         genomic            59
invertebrate   chloroplast         1
Name: count, dtype: int64

One combination stands out: there's a single observation of an organism with 'Kingdom' being 'invertebrate', but 'DNAtype' being 'chloroplast' - to the best of my knowledge, chloroplasts are exclusive to photosynthetic life, and an invertebrate shouldn't have any.  What's going on here?

In [8]:
toxo = (codon_usage['kingdom'] == 'invertebrate') & (codon_usage['dna_type'] == 'chloroplast')
codon_usage.loc[toxo, ['Kingdom', 'DNAtype', 'SpeciesID', 'Ncodons', 'SpeciesName']]

,Kingdom,DNAtype,SpeciesID,Ncodons,SpeciesName
9400,inv,2,5811,6158,chloroplast Toxoplasma gondii


'chloroplast Toxoplasma gondii' is most likely the apicoplast present in T. gondii which is believed to be a vestige of a chloroplast in the evolutionary history of several organisms.  (See https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3166169/)  It is not chloroplast DNA, and a single row cannot be modelled as a class of its own, so it is dropped.

In [9]:
# a boolean mask, not a hard-coded row label
codon_usage = codon_usage[~toxo]
codon_usage.shape

(12963, 72)

In [10]:
codon_usage['kingdom'].value_counts()

kingdom
bacteria         2918
virus            2832
plant            2490
vertebrate       2077
invertebrate     1333
mammal            572
bacteriophage     220
rodent            215
primate           180
archaea           126
Name: count, dtype: int64

In [11]:
codon_usage['dna_type'].value_counts()

dna_type
genomic          9249
mitochondrial    2899
chloroplast       815
Name: count, dtype: int64

Ten kingdoms and three DNA types remain. Check for missing values:

In [12]:
codon_usage.isna().sum().sum()

np.int64(0)

None. In 2022 there were two at this point: DNA-type code 9 had been written to a CSV as the string 'NA' and read back as missing, and `dropna()` removed the two rows without comment. Here they were removed, and counted, in the DNA-type step above. `codon_utils.filter_classes` applies the same three steps and logs the row count after each:

In [13]:
model_rows, steps = codon_utils.filter_classes(codon_utils.read_raw('../data/raw/codon_usage.csv'))
assert model_rows.index.equals(codon_usage.index)  # same rows as the steps above
steps

,step,rows,removed
0,raw,13028,0
1,drop Kingdom 'plm' (plasmid),13010,18
2,"keep DNAtype in {0 genomic, 1 mitochondrial, 2...",12964,46
3,drop inv + chloroplast (Toxoplasma apicoplast)...,12963,1


## Scaling happens inside cross-validation

k-NN and the MLP depend on the scale of the features (distances and gradient steps), tree models do not. A scaler fitted on all rows would let the test rows shape the transformation applied to the training rows, so in `04_modeling` k-NN and the MLP are pipelines whose `StandardScaler` is fitted on the training folds only.

## Features

The two feature sets compared in `04_modeling`: the 64 codon frequencies, and the 21 amino-acid frequencies built in `01_wrangling`.

In [14]:
amino = codon_utils.amino_features(codon_usage)
print('codon features:', len(codon_utils.CODONS), ' amino-acid features:', amino.shape[1])

codon features: 64  amino-acid features: 21


In [15]:
amino.mean()

alanine          0.074558
arginine         0.046394
asparagine       0.044484
aspartic_acid    0.045354
cysteine         0.014892
glutamine        0.034860
glutamic_acid    0.049968
glycine          0.065275
histidine        0.023725
isoleucine       0.071649
leucine          0.108341
lysine           0.049952
methionine       0.021139
phenylalanine    0.048169
proline          0.049957
serine           0.071234
threonine        0.062768
tryptophan       0.011609
tyrosine         0.034326
valine           0.062914
stop             0.008432
dtype: float64

In [16]:
amino.std()

alanine          0.024743
arginine         0.019463
asparagine       0.014037
aspartic_acid    0.017570
cysteine         0.010464
glutamine        0.014233
glutamic_acid    0.019886
glycine          0.017715
histidine        0.007687
isoleucine       0.032335
leucine          0.036655
lysine           0.021792
methionine       0.008158
phenylalanine    0.017158
proline          0.014652
serine           0.015627
threonine        0.019276
tryptophan       0.006574
tyrosine         0.010434
valine           0.015594
stop             0.011055
dtype: float64

These are raw frequencies, not standard scores: the means run from 0.008 (stop) to 0.108 (leucine) and the standard deviations from 0.007 to 0.037. (The 2022 text said they had been standardized, but the scaler had been commented out.)

## Labels

Labels stay as strings. The cross-validation harness in `04_modeling` encodes them as integers for every model. In 2022, k-NN and random forest were fitted on one-hot target columns (`pd.get_dummies`), which scikit-learn treats as ten separate yes/no problems: a row could receive no label at all, and those rows counted as errors (see `ERRATUM.md`).

## Groups for cross-validation

Related rows must not be split between training and test folds, or the test score measures how well a model recognises near-copies rather than new organisms. Two levels of grouping are used: *species*, the `SpeciesID` (the NCBI taxonomy id); and *genus*, approximated by the first informative word of the species name (`codon_utils.name_token`), made constant within each `SpeciesID` so that a genus group always contains whole species.

In [17]:
groups = codon_utils.add_groups(codon_usage)
print('species groups:', groups['group_species'].nunique(), ' genus groups:', groups['group_genus'].nunique())
groups['group_genus'].value_counts().head()

species groups: 12331  genus groups: 4548


group_genus
human             388
streptomyces      186
tomato            154
pseudomonas       150
enterobacteria    146
Name: count, dtype: int64

The genus heuristic is rough for viruses: the largest group, 'human', collects every virus whose name starts with "Human". Because `SpeciesID` is the NCBI taxonomy id, real genus, family and order groups can be built from the NCBI taxonomy instead (see the README).

What do rows that share a `SpeciesID` have in common?

In [18]:
multi = groups[groups['SpeciesID'].duplicated(keep=False)]
print(multi['SpeciesID'].nunique(), 'SpeciesIDs have more than one row;',
      'rows that repeat a (SpeciesID, DNA type) pair:', groups.duplicated(['SpeciesID', 'DNAtype']).sum())
multi.groupby('SpeciesID')['dna_type'].agg(lambda s: ' + '.join(sorted(s))).value_counts()

596 SpeciesIDs have more than one row; rows that repeat a (SpeciesID, DNA type) pair: 0


dna_type
genomic + mitochondrial                  423
chloroplast + genomic                    129
chloroplast + genomic + mitochondrial     36
chloroplast + mitochondrial                8
Name: count, dtype: int64

They are always different compartments of one organism. Are their codon profiles similar? Compare the L1 distance between two rows with the same `SpeciesID` to the distance between random pairs of rows:

In [19]:
X = groups[codon_utils.CODONS].to_numpy()
position = pd.Series(np.arange(len(groups)), index=groups.index)
same = [np.abs(X[position[g.index[0]]] - X[position[g.index[1]]]).sum()
        for _, g in multi.groupby('SpeciesID')]
rng = np.random.default_rng(0)
a, b = rng.integers(0, len(X), 20_000), rng.integers(0, len(X), 20_000)
random_pairs = np.abs(X[a[a != b]] - X[b[a != b]]).sum(axis=1)
print(f'median L1 distance: same SpeciesID {np.median(same):.3f} ({len(same)} pairs), '
      f'random rows {np.median(random_pairs):.3f} ({len(random_pairs):,} pairs)')

median L1 distance: same SpeciesID 0.771 (596 pairs), random rows 0.699 (19,999 pairs)


Two rows of one organism are no closer than two random rows: a genomic and a mitochondrial codon profile differ about as much as two unrelated organisms do. So a `SpeciesID` shared between training and test folds should leak little; `04_modeling` measures it. Relatedness between *different* `SpeciesID`s is another matter. Strains and close relatives are often filed under separate ids, some with identical codon vectors:

In [20]:
key = groups[codon_utils.CODONS].round(5).astype(str).agg('|'.join, axis=1)
twins = groups[key.duplicated(keep=False)].assign(vector=key)
by_vector = twins.groupby('vector')
print(len(twins), 'rows form', by_vector.ngroups, 'groups of identical codon vectors;',
      (by_vector['SpeciesID'].nunique() > 1).sum(), 'span several SpeciesIDs and',
      (by_vector['group_genus'].nunique() == 1).sum(), 'lie within one genus group')
by_vector['SpeciesName'].agg(' | '.join).head(6).to_frame().reset_index(drop=True)

63 rows form 23 groups of identical codon vectors; 23 span several SpeciesIDs and 20 lie within one genus group


,SpeciesName
0,Bordetella hinzii BC-306 | Bordetella hinzii L...
1,Sphingomonas sp. GTIN11 | Klebsiella sp. LSSE-H2
2,Measles virus strain Moraten | Measles virus s...
3,Salmonella phage SETP7 | Salmonella phage SETP11
4,Cyanophage Ma-LMM02 | Cyanophage Ma-LMM03 | Cy...
5,Norovirus Hu/NLV/Oxford/B4S2/2002/UK | Norovir...


Identical vectors under different ids, such as two *Bordetella hinzii* strains or the Moraten and Schwarz measles vaccine strains, put near-copies on both sides of a species-level split, but mostly not of a genus-level split. `04_modeling` therefore evaluates three fold schemes: random folds, species held out, and genus held out.

`04_modeling` rebuilds this frame with `codon_utils.load_project` (`read_raw`, then `filter_classes`, then `add_groups`). Check that it gives the same rows and groups:

In [21]:
project, _ = codon_utils.load_project('../data/raw/codon_usage.csv')
assert project.index.equals(groups.index)
assert (project['group_genus'] == groups['group_genus']).all()
print('load_project matches this notebook:', len(project), 'rows')

load_project matches this notebook: 12963 rows
